# Unity Catalog Configuration

## Overview
This notebook sets up the data infrastructure for a the parse_ai_documents function demo by:
1. Importing Unity Catalog configuration settings
2. Creating a volume for raw data storage
3. Loading sample_files to observe the competences of the function

## 1- Set the UC schema

In [0]:
%run ../_config/config_unity_catalog

## 2- Create a Volume

### What is a Volume?
A Unity Catalog Volume is a storage location for unstructured data (files).

### Purpose
In this volume, we will load CSV and PDF files before creating Delta tables.

### Benefits
- Governed access control through Unity Catalog
- Centralized file management
- Version control and lineage tracking
- Integration with Databricks File System (DBFS)

In [0]:
%sql
-- Create a Unity Catalog volume named 'raw_data'
-- IF NOT EXISTS: Prevents errors if the volume already exists
-- This volume will be created in the current catalog.schema namespace
-- Location: /Volumes/{catalog}/{schema}/raw_data/
CREATE VOLUME IF NOT EXISTS raw_data;

## Load sources Files

### Source 

Main branch of the demo repository

### Process
Uses `dbutils.fs.cp()` to copy files from GitHub to the Unity Catalog volume.

In [0]:
# Define the source location (GitHub raw content URL)
# This points to the main branch of the demo repository
prefix = "https://github.com/O-Faraday/databricks_genai_demo/raw/main/"

# Define the destination path in the Unity Catalog volume
# Uses the catalog and schema variables imported from the config notebook
# Format: /Volumes/{catalog}/{schema}/raw_data/
path_volume = f"/Volumes/{catalog}/{schema}/raw_data/"

# List of PDF file names to download
# These files contain knowledge base information for the GenAI agent
l_sources_names = [
    "sample_dashboard.png",    # Sample dashboard screenshot
    "old_articles.pdf",      # Old scanned newspapers
    "Invoice.jpg", # from landing ai
    "AccidentStatement.pdf" # from landing ai
]

# Loop through each PDF file and copy it to the volume
for source_name in l_sources_names:
    # Construct source URL: repository URL + data/pdf/ + filename.pdf
    source_path = f"{prefix}data/multiformat/{source_name}"
    # Construct destination path: volume path + pdf/ + filename.pdf
    destination_path = f"{path_volume}/multiformat/{source_name}"
    
    dbutils.fs.cp(
        source_path,
        destination_path
    )
    
